In [4]:
import pandas as pd

DATA_PATH = "../../data/combined_data.csv"

data = pd.read_csv(DATA_PATH)
data.head()

,id,province,city,district,month_sales,lat,lng,hours
0,8388608,江苏省,南京市,鼓楼区,13.0,22.713545,115.865425,23:29-23:59
1,8388612,辽宁省,葫芦岛市,龙港区,111.0,40.718140,120.856560,09:00-23:00
2,8388613,广东省,深圳市,宝安区,46.0,22.732786,113.817619,10:00-22:00
3,8388614,内蒙古自治区,呼和浩特市,赛罕区,13.0,40.694965,111.761520,10:00-21:10
4,8388615,广东省,东莞市,东坑镇,1375.0,22.983700,113.940262,09:14-23:35


In [ ]:
tmp_data = df[df["city"] == city_name_cn]
print(f"{city_name_cn} has {len(tmp_data)} entries.")

min_lat = boundaries_dict["min_lat"]
max_lat = boundaries_dict["max_lat"]
min_lng = boundaries_dict["min_lng"]
max_lng = boundaries_dict["max_lng"]

print(f"Using hardcoded boundaries:")
print(f"  Latitude: {min_lat} to {max_lat}")
print(f"  Longitude: {min_lng} to {max_lng}")

valid_data = tmp_data[
    tmp_data["lat"].notna()
    & tmp_data["lng"].notna()
    & tmp_data["month_sales"].notna()
    & (tmp_data["lat"] >= min_lat)
    & (tmp_data["lat"] <= max_lat)
    & (tmp_data["lng"] >= min_lng)
    & (tmp_data["lng"] <= max_lng)
].copy()

print(f"After filtering, {city_name_cn} has {len(valid_data)} valid entries.\n")

In [ ]:
# draw the distribution of month_sales
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

# log scale for x-axis
plt.xscale('log')
plt.hist(shenzhen_data['month_sales'], bins=30, color='skyblue', edgecolor='black')
plt.title('The Distribution of Monthly Sales in Shenzhen')
plt.xlabel('Monthly Sales')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.savefig("figure/shenzhen_month_sales_distribution.png")

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
import contextily as ctx
from matplotlib.colors import LinearSegmentedColormap

# Define Shenzhen city boundaries (excluding Shenshan Special Cooperation Zone)
# Latitude: approx 22.4 to 22.9
# Longitude: approx 113.75 to 114.65 (Dapeng ends around 114.65)
min_lat, max_lat = 22.4, 22.9
min_lng, max_lng = 113.75, 114.65

print(f"Using hardcoded Shenzhen boundaries:")
print(f"  Latitude: {min_lat} to {max_lat}")
print(f"  Longitude: {min_lng} to {max_lng}")

# Filter data within Shenzhen city boundaries
valid_data = shenzhen_data[
    shenzhen_data['lat'].notna() & 
    shenzhen_data['lng'].notna() & 
    shenzhen_data['month_sales'].notna() &
    (shenzhen_data['lat'] >= min_lat) &
    (shenzhen_data['lat'] <= max_lat) &
    (shenzhen_data['lng'] >= min_lng) &
    (shenzhen_data['lng'] <= max_lng)
].copy()

print(f"\nFiltered data count: {len(valid_data)} (Original: {len(shenzhen_data)})")
print(f"Actual data latitude range: {valid_data['lat'].min():.4f} - {valid_data['lat'].max():.4f}")
print(f"Actual data longitude range: {valid_data['lng'].min():.4f} - {valid_data['lng'].max():.4f}")

lat = valid_data['lat'].values
lng = valid_data['lng'].values
sales = valid_data['month_sales'].values

# Calculate aspect ratio to maintain geographic proportions
# At latitude ~22.6°, 1 degree longitude ≈ 1 degree latitude * cos(22.6°)
avg_lat = np.mean(lat)
aspect_ratio = 1.0 / np.cos(np.radians(avg_lat))

# Calculate suitable figsize based on data bounds
# Add padding to ranges as used in set_xlim/ylim later (0.02 * 2 = 0.04)
padding = 0.04
lat_range = (lat.max() - lat.min()) + padding
lng_range = (lng.max() - lng.min()) + padding

# Height needs to be scaled by aspect_ratio to match visual proportions
plot_height = lat_range * aspect_ratio
plot_width = lng_range

# Set base width and calculate height
base_width = 15
fig_height = base_width * (plot_height / plot_width)

# Create figure
fig, ax = plt.subplots(figsize=(base_width, fig_height), dpi=150)

# Create weighted 2D histogram (weighted by sales)
# Increase resolution (approx 5x previous)
bins_x = 1000
bins_y = int(bins_x * (plot_height / plot_width))

# Generate weighted 2D histogram
heatmap, xedges, yedges = np.histogram2d(
    lng, lat, 
    bins=[bins_x, bins_y],
    weights=sales
)

# Apply Gaussian filter to spread heat around each point
# Larger sigma value means wider spread
# Scaled sigma for higher resolution (approx 5x bins -> 5x sigma)
heatmap_smooth = gaussian_filter(heatmap, sigma=15)

# Mask the background (low values) to make them transparent
# Use a very low threshold so the alpha gradient handles the fade
mask_threshold = heatmap_smooth.max() * 0.001
heatmap_masked = np.ma.masked_where(heatmap_smooth < mask_threshold, heatmap_smooth)

# Create custom colormap with transparency gradient
# Base colormap: Spectral_r (Blue -> Green -> Yellow -> Red)
ncolors = 256
color_array = plt.cm.Spectral_r(np.linspace(0, 1, ncolors))

# Create alpha gradient
# "Rapid decay": Alpha should be high (1.0) for most values, but drop quickly to 0 at the low end.
# Below 15% of the range, it fades from 0 to 1.0. Above 15%, it stays fully opaque (1.0).
threshold_idx = int(ncolors * 0.10)
alphas = np.concatenate((
    np.linspace(0.0, 1.0, threshold_idx),           # Rapid rise from 0 to 1.0
    np.ones(ncolors - threshold_idx)                # Stable high opacity (1.0)
))
color_array[:, -1] = alphas

# Create new colormap
map_cmap = LinearSegmentedColormap.from_list(name='Spectral_r_alpha', colors=color_array)

# Plot heatmap
extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
im = ax.imshow(
    heatmap_masked.T,
    extent=extent,
    origin='lower',
    cmap=map_cmap,  # Use custom transparent colormap
    aspect=aspect_ratio,  # Maintain geographic proportions
    interpolation='bilinear',
    zorder=10  # Ensure heatmap is on top of basemap
)

# Add basemap with administrative boundaries
try:
    # Add basemap
    ctx.add_basemap(
        ax,
        crs='EPSG:4326',
        source=ctx.providers.OpenStreetMap.Mapnik,
        alpha=1.0, # Map fully opaque (or slightly transparent if desired)
        zorder=0   # Map at bottom
    )
    print("\nBasemap added successfully")
except Exception as e:
    print(f"Could not add basemap: {e}")

# Add colorbar
cbar = plt.colorbar(im, ax=ax, label='Sales Density (Weighted)')
cbar.ax.tick_params(labelsize=10)

# Set title and labels
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Shenzhen Food Delivery Heatmap by Monthly Sales', fontsize=16, fontweight='bold', pad=20)

# Add grid
ax.grid(True, alpha=0.3, linestyle='--', color='white')

# Add statistics
stats_text = f'Total Stores: {len(valid_data)}\nAvg Sales: {sales.mean():.1f}\nMax Sales: {sales.max():.0f}'
ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Set axis limits
ax.set_xlim(lng.min() - 0.02, lng.max() + 0.02)
ax.set_ylim(lat.min() - 0.02, lat.max() + 0.02)

plt.tight_layout()
plt.savefig('figure/shenzhen_heatmap.png', dpi=300, bbox_inches='tight')
print("\nHeatmap saved as figure/shenzhen_heatmap.png")
plt.show()